# **Lo**w-**R**ank **A**dapater

大模型由于参数量大，微调训练消耗较多资源，这种微调方式称之为全参微调(full-parameter Fintuning)。为了达到高效训练的目的，常见的方式为冻结模型特定层，仅训练特定层，如输出头。

一类高效微调方法是在原模型基础上，附着一个旁路网络，仅微调旁路网络，性能逼近全参微调效果，这一类方法称为参数高效微调(Parameters Efficient FineTunning, PEFT)

LoRA（Low-Rank Adapter） 微调方法，在原模型基础上，增加旁路分支，而分支参数量原小于原模型。例如一个原参数矩阵为 $W\in\mathbb{R}^{d\times d}$ 增加一个网络参数量 $|\theta|<<d\times d$。一个线性层的前向变为 $XW+X\theta$.

本 lecture 按照如下顺序讲解 LoRA：

1. LoRA 原理及推导
2. LoRA 实现
3. 为什么 LoRA 能work? SVD 分解？
4. 原参数梯度矩阵是否有低秩特性？
5. 如何取 Rank？
6. LoRA 的本质与梯度子空间相似性度量
7. LoRA 与 全参微调对比
8. LoRA 微调过程显存分析
9. QLoRA 实现

## 推导

给定线性投影模型:
$$
h=XW
$$
其中, $X\in\mathbb{R}^{N \times d_\text{in}}$, $W\in\mathbb{R}^{d_\text{in} \times d_\text{out}}$, 其梯度为 $\triangle W$ , 模型更新为：
$$
W'=W+\triangle W
$$
此时前向过程为:
$$
h'=XW'=X(W+\triangle W)=XW+X\triangle W
$$
给定最优梯度为$\triangle W^*$
$$
h'=XW'=X(W+\triangle W^*)=XW+X\triangle W^*
$$
将原有的目标求最优:
$$
\arg\min_W \mathcal{L}(W)
$$
转化为求最优梯度，原参数$W$冻结
$$
\arg\min_{\triangle W} \mathcal{L}(W+ \triangle W)
$$
此时将最优梯度参数化为 $\delta\in\mathbb{R}^{d_\text{in}\times d_\text{out}}$
$$
\arg\min_{\delta} \mathcal{L}(W+ \delta)
$$
在优化 $\delta$ 参数量与原模型是一样的，整体网络计算量更大，此时将 $\delta$ 转化为
$$
\delta \rightarrow W_AW_B
$$
其中$W_A\in \mathbb{R}^{d_\text{in}\times r},W_B\in \mathbb{R}^{r \times d_\text{out}}$, $r\ll d$ 为秩, 此时前向计算为
$$
h'=XW'=X(W+\triangle W^*)=XW+X \alpha W_AW_B
$$
其中$\alpha$为超参数,控制旁路分支强度,优化目标为:
$$
W_A^*,W_B^*=\arg\min_{W_A,W_B} \mathcal{L}(W, W_AW_B)
$$


## 实现

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [36]:
dim = 512
r = 4
bsz = 1
seq_len = 128
vocab_size = 100

x = torch.randint(vocab_size, (bsz, seq_len))
X = torch.randn(bsz, seq_len, dim)

In [45]:
class Attention(nn.Module):
    def __init__(self, dim_in, dim_out):
        super().__init__()
        self.Wq = nn.Linear(dim_in, dim_out, bias=False)
        self.Wk = nn.Linear(dim_in, dim_out, bias=False)
        self.Wv = nn.Linear(dim_in, dim_out, bias=False)
        self.Wo = nn.Linear(dim_out, dim_out, bias=False)
    def forward(self, X):
        Q, K, V = self.Wq(X), self.Wk(X), self.Wv(X)
        S = Q@K.transpose(1,2)
        P = F.softmax(S, dim = -1)
        Z = P @ V
        O = self.Wo(Z)
        return O

model = Attention(dim, dim)
model(X)
print(model)

Attention(
  (Wq): Linear(in_features=512, out_features=512, bias=False)
  (Wk): Linear(in_features=512, out_features=512, bias=False)
  (Wv): Linear(in_features=512, out_features=512, bias=False)
  (Wo): Linear(in_features=512, out_features=512, bias=False)
)


In [46]:
W = nn.Linear(2,3)
W.in_features

2

In [47]:
class LoRALinear(nn.Module):
    def __init__(self, original_linear, rank=4, alpha=0.1):
        super().__init__()
        self.alpha = alpha
        self.dim_in = original_linear.in_features
        self.dim_out = original_linear.out_features
        self.r=rank

        # 参数
        self.weight = nn.Parameter(original_linear.weight.data.clone(), requires_grad=False)
        # 处理偏置
        if original_linear.bias is not None:
            self.bias = nn.Parameter(original_linear.bias.data.clone(), requires_grad=False)
        else:
            self.register_parameter('bias', None)
        
        self.WA = nn.Linear(self.dim_in, self.r)
        self.WB = nn.Linear(self.r, self.dim_out)

    def forward(self, X):
        # 出错
        # h = F.linear(x, self.weight, self.bias)

        # 手动
        bsz, seq_len, dim_in =X.shape
        h = X.view(bsz*seq_len, dim_in) @ self.weight
        h = h.view(bsz, seq_len, dim_in)
        if self.bias is not None:
            h += bias.unsqueeze(dim=0).unsqueeze(dim=0)
        
        h_lora = self.alpha * self.WB(self.WA(X))
        return h_lora + h

In [48]:
def apply_lora_adapter(model, MyLinear, r):
    """
    更通用的替换函数，支持带偏置的 Linear 层
    """
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            new_layer = MyLinear(module, r)
            setattr(model, name, new_layer)
            print(f"Replaced {name}")
        else:
            # 递归替换
            replace_linear_layers_advanced(module)

In [49]:
model = Attention(dim, dim)
print(model)
apply_lora_adapter(model, LoRALinear, r=4)
print(model)

H = model(X)
print(h.shape)

Attention(
  (Wq): Linear(in_features=512, out_features=512, bias=False)
  (Wk): Linear(in_features=512, out_features=512, bias=False)
  (Wv): Linear(in_features=512, out_features=512, bias=False)
  (Wo): Linear(in_features=512, out_features=512, bias=False)
)
Replaced Wq
Replaced Wk
Replaced Wv
Replaced Wo
Attention(
  (Wq): LoRALinear(
    (WA): Linear(in_features=512, out_features=4, bias=True)
    (WB): Linear(in_features=4, out_features=512, bias=True)
  )
  (Wk): LoRALinear(
    (WA): Linear(in_features=512, out_features=4, bias=True)
    (WB): Linear(in_features=4, out_features=512, bias=True)
  )
  (Wv): LoRALinear(
    (WA): Linear(in_features=512, out_features=4, bias=True)
    (WB): Linear(in_features=4, out_features=512, bias=True)
  )
  (Wo): LoRALinear(
    (WA): Linear(in_features=512, out_features=4, bias=True)
    (WB): Linear(in_features=4, out_features=512, bias=True)
  )
)
torch.Size([1, 128, 512])


In [50]:
label = torch.randn(bsz, seq_len, dim)
loss_fn = nn.MSELoss()
loss = loss_fn(h, label)
print(loss)
loss.backward()

tensor(1.0024, grad_fn=<MseLossBackward0>)


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [44]:
for n, p in model.named_parameters():
    print(n, p.grad)

Wq.weight None
Wq.WA.weight None
Wq.WA.bias None
Wq.WB.weight None
Wq.WB.bias None
Wk.weight None
Wk.WA.weight None
Wk.WA.bias None
Wk.WB.weight None
Wk.WB.bias None
Wv.weight None
Wv.WA.weight None
Wv.WA.bias None
Wv.WB.weight None
Wv.WB.bias None
Wo.weight None
Wo.WA.weight None
Wo.WA.bias None
Wo.WB.weight None
Wo.WB.bias None


## 为什么 LoRA 能work? SVD 分解？

## 原参数梯度矩阵是否有低秩特性？

## 如何取 Rank？

## LoRA 的本质与梯度子空间相似性度量

## LoRA 与 全参微调对比

## LoRA 微调过程显存分析

## QLoRA 实现